In [4]:
import os
import cv2
import numpy as np

caminho_video = "videos/video_3.mp4"
if not os.path.exists(caminho_video):
    caminho_video = "21249383.mp4"

cap = cv2.VideoCapture(caminho_video)
if not cap.isOpened():
    raise FileNotFoundError(f"Erro ao abrir vídeo: {caminho_video}")

largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0:
    fps = 30

# Gravador de vídeo de saída
caminho_saida = "resultado_esteira_contagem1.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(caminho_saida, fourcc, fps, (largura, altura))

# 1. Subtrator de fundo configurado para ignorar sombras fracas
subtrator = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=40, detectShadows=True)

# 2. Elementos estruturantes (Morfologia Matemática)
kernel_abertura = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_fechamento = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))

# 3. Linha de Contagem (70% da altura da tela, próximo ao final da esteira)
linha_y = int(altura * 0.70)
offset_faixa = 12  # Faixa de tolerância de cruzamento

total_caixas = 0
caixas_ativas = []  # Guarda os centróides (cx, cy) dos objetos detectados

print(f"Processando vídeo '{caminho_video}'...")
frame_idx = 0

while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    frame_idx += 1

    # Pré-processamento
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    suavizado = cv2.GaussianBlur(cinza, (5, 5), 0)

    # Subtração de fundo
    mascara = subtrator.apply(suavizado)

    # Limiarização para remover sombras (pixels cinzas do MOG2 são marcados como 127)
    _, mascara_bin = cv2.threshold(mascara, 200, 255, cv2.THRESH_BINARY)

    # Morfologia Matemática:
    # Abertura: remove ruídos e pequenas cintilações nos roletes
    mascara_limpa = cv2.morphologyEx(mascara_bin, cv2.MORPH_OPEN, kernel_abertura)
    # Fechamento: une as bordas e preenche o interior das caixas
    mascara_limpa = cv2.morphologyEx(mascara_limpa, cv2.MORPH_CLOSE, kernel_fechamento)

    # Extração de contornos e Bounding Boxes
    contornos, _ = cv2.findContours(mascara_limpa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    novos_centroides = []

    for contorno in contornos:
        area = cv2.contourArea(contorno)
        
        # Filtra por área mínima para ignorar pequenas interferências e sujeiras
        if area > 1200:
            x, y, w, h = cv2.boundingRect(contorno)
            cx = x + w // 2
            cy = y + h // 2

            # Desenha a Bounding Box e o centróide
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            novos_centroides.append((cx, cy))

            # Lógica de Cruzamento: verifica se o centróide está cruzando a linha
            if (linha_y - offset_faixa) < cy < (linha_y + offset_faixa):
                # Confere se este objeto já não foi contabilizado por proximidade
                ja_contado = False
                for prev_cx, prev_cy in caixas_ativas:
                    if abs(cx - prev_cx) < 45 and abs(cy - prev_cy) < 30:
                        ja_contado = True
                        break

                if not ja_contado:
                    total_caixas += 1
                    cv2.line(frame, (0, linha_y), (largura, linha_y), (0, 255, 255), 4)

    caixas_ativas = novos_centroides

    # Desenho da Linha de Contagem e do Painel
    cv2.line(frame, (0, linha_y), (largura, linha_y), (0, 0, 255), 2)
    cv2.putText(frame, "LINHA DE CONTAGEM", (20, linha_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.rectangle(frame, (15, 20), (360, 85), (0, 0, 0), -1)
    cv2.putText(frame, f"TOTAL CAIXAS: {total_caixas}", (30, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    out.write(frame)

    if frame_idx % 90 == 0:
        print(f"Processando frame {frame_idx}... Caixas contabilizadas: {total_caixas}")

cap.release()
out.release()
print(f"\nConcluído! Total de caixas contabilizadas: {total_caixas}")
print(f"Vídeo com as detecções salvo em: {os.path.abspath(caminho_saida)}")

Processando vídeo 'videos/video_3.mp4'...
Processando frame 90... Caixas contabilizadas: 13
Processando frame 180... Caixas contabilizadas: 23
Processando frame 270... Caixas contabilizadas: 40
Processando frame 360... Caixas contabilizadas: 59
Processando frame 450... Caixas contabilizadas: 71

Concluído! Total de caixas contabilizadas: 79
Vídeo com as detecções salvo em: C:\Users\fabri\Documents\pythonScripts\Aula 24\resultado_esteira_contagem1.mp4
